# Pipeline End-to-End — Node-by-Node Trace

Notebook chay toan bo pipeline EduBot, phan ro **tung node** de kiem soat:

```
1 ContextAnalyzer -> 2 IntentRouter (LLM) -> 3 SessionManager -> 4 ActionPlanner -> 5 RAG Search -> 6 Handler -> 7 Session Save
```

Ket qua pipeline duoc ghi ra `pipeline_trace.log` dang **JSON** (reset moi query). File `app.log` giu persistent log.

---
## 0. Setup — Path & Environment

In [1]:
import sys
import os
import time
import json
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent.parent
print(f"Project root: {PROJECT_ROOT}")

# Add src to path
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Load env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

print(f"API Key: {'SET' if os.getenv('GENAI_API_KEY') else 'NOT SET'}")
print(f"Python: {sys.version.split()[0]}")

Project root: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\ĐATN
API Key: SET
Python: 3.12.4


---
## 1. Init — CustomSearch + Reranker + Orchestrator

In [2]:
from src.config.config import settings
from src.rag.retrieve_rebuild import CustomSearch
from src.rag.reranker import Reranker
from src.llm.orchestrator import Orchestrator

DATA_DIR = PROJECT_ROOT / 'data'
CHUNKS_PATH = str(DATA_DIR / 'rag_chunks_v2.json')
EMBEDDINGS_PATH = str(DATA_DIR / 'embeddings.npy')

print("=" * 60)
print("[NODE 0] Initializing Components")
print("=" * 60)

# 1a. CustomSearch
t0 = time.time()
searcher = CustomSearch(chunks_path=CHUNKS_PATH, embeddings_path=EMBEDDINGS_PATH)
print(f"  CustomSearch: {searcher.corpus_size} chunks, dim={searcher.embeddings.shape[1]} ({time.time()-t0:.2f}s)")

# 1b. Reranker
reranker = Reranker()
print(f"  Reranker: {settings.RERANKER_MODEL} (lazy load)")

# 1c. Orchestrator
orch = Orchestrator(retriever=searcher, reranker=reranker)
print(f"  Orchestrator: ready")
print(f"  LLM Model: {settings.LLM_MODEL}")
print("=" * 60)

[NODE 0] Initializing Components
CustomSearch initialized: 2348 docs, vocab=17139, avgdl=140.9
  CustomSearch: 2348 chunks, dim=768 (0.15s)
  Reranker: AITeamVN/Vietnamese_Reranker (lazy load)
  Orchestrator: ready
  LLM Model: gemini-2.5-flash-lite


---
## 2. Helper — Log Viewer & Node Runner

In [3]:
TRACE_LOG = PROJECT_ROOT / 'logs' / 'pipeline_trace.log'
APP_LOG = PROJECT_ROOT / 'logs' / 'app.log'

def show_trace_log():
    """Hien thi pipeline_trace.log (JSON format)."""
    if TRACE_LOG.exists():
        content = TRACE_LOG.read_text(encoding='utf-8')
        try:
            data = json.loads(content)
            print(json.dumps(data, indent=2, ensure_ascii=False, default=str))
        except json.JSONDecodeError:
            print(content)  # fallback to raw text
    else:
        print('[!] pipeline_trace.log chua ton tai')

def load_trace_json() -> dict:
    """Load pipeline_trace.log as dict."""
    if TRACE_LOG.exists():
        return json.loads(TRACE_LOG.read_text(encoding='utf-8'))
    return {}

def show_app_log(n=30):
    """Hien thi n dong cuoi cua app.log."""
    if APP_LOG.exists():
        lines = APP_LOG.read_text(encoding='utf-8').strip().split('\n')
        for line in lines[-n:]:
            print(line)
    else:
        print('[!] app.log chua ton tai')

def show_debug_info(debug_info: dict):
    """Hien thi debug info tu orchestrator.last_debug_info."""
    if not debug_info:
        print('[!] Chua co debug info')
        return
    print(json.dumps(debug_info, indent=2, ensure_ascii=False, default=str))

print('Helpers loaded: show_trace_log(), load_trace_json(), show_app_log(n), show_debug_info(info)')

Helpers loaded: show_trace_log(), load_trace_json(), show_app_log(n), show_debug_info(info)


---
## 3. Chay tung Node rieng le

Phan nay tach pipeline ra tung buoc de debug.

### 3.1 — Node: ContextAnalyzer

In [4]:
from src.llm.context_analyzer import ContextAnalyzer

analyzer = ContextAnalyzer()

# Test cases
test_cases = [
    {"query": "Mang LAN la gi?", "history": ""},
    {"query": "No khac gi voi WAN?", "history": "user: Mang LAN la gi?\nassistant: Mang LAN la..."},
    {"query": "Sinh 3 cau hoi trac nghiem", "history": "user: Giai thich TCP/IP\nassistant: TCP/IP la..."},
    {"query": "ok", "history": "user: Co dung khong?\nassistant: Dung roi"},
]

print("=" * 60)
print("[NODE 1] ContextAnalyzer")
print("=" * 60)
for tc in test_cases:
    needs = analyzer.needs_contextualization(tc["query"], tc["history"])
    print(f"  Query: '{tc['query'][:50]}'")
    print(f"  History: {'Yes' if tc['history'] else 'No'}")
    print(f"  Needs context: {needs}")
    if needs and tc['history']:
        snippet = analyzer.extract_context_from_history(tc['history'])
        print(f"  Extracted: '{snippet[:100]}'")
    print()

[NODE 1] ContextAnalyzer
  Query: 'Mang LAN la gi?'
  History: No
  Needs context: False

  Query: 'No khac gi voi WAN?'
  History: Yes
  Needs context: False

  Query: 'Sinh 3 cau hoi trac nghiem'
  History: Yes
  Needs context: False

  Query: 'ok'
  History: Yes
  Needs context: True
  Extracted: 'user: Co dung khong?
assistant: Dung roi'



### 3.2 — Node: IntentRouter (LLM Call)

In [ ]:
from src.llm.intent_router import IntentRouter

router = IntentRouter()

test_queries = [
    "Tao 3 cau trac nghiem ve mang may tinh",
    "Giai thich TCP/IP la gi",
    "Xin chao ban la ai",
    "Tao slide bai an toan thong tin",
    "Cau 1 dap an A",
]

print("=" * 60)
print("[NODE 2] IntentRouter (LLM Calls)")
print("=" * 60)
for q in test_queries:
    t0 = time.time()
    result = router.detect(query=q)
    elapsed = time.time() - t0
    print(f"  Query: '{q}'")
    print(f"  Intent: {result.primary_intent} | Task: {result.task_type} | Topic: {result.topic} | New: {result.is_new_topic}")
    print(f"  Time: {elapsed:.2f}s")
    print()

### 3.3 — Node: SessionManager

In [ ]:
from src.llm.intent_router import IntentResult
from src.llm.session_manager import SessionManager
from src.llm.session_store import SessionStore
from src.llm.memory import MemoryManager

memory = MemoryManager()
store = SessionStore(storage_path=str(PROJECT_ROOT / 'data' / 'sessions'))
sm = SessionManager(session_store=store, memory=memory)

# Simulate intent results
intents = [
    IntentResult(primary_intent="chat", topic=None, is_new_topic=True),
    IntentResult(primary_intent="generate", task_type="mcq", topic="Mang may tinh", is_new_topic=True),
    IntentResult(primary_intent="interact", task_type=None, topic="Mang may tinh", is_new_topic=False),
]

print("=" * 60)
print("[NODE 3] SessionManager")
print("=" * 60)
for i, intent in enumerate(intents):
    session = sm.resolve_session(intent)
    print(f"  Step {i+1}: intent={intent.primary_intent}, topic={intent.topic}, new={intent.is_new_topic}")
    print(f"    -> Session: {session.session_id}, topic='{session.topic}', msgs={len(session.messages)}")
    print()

### 3.4 — Node: ActionPlanner

In [ ]:
from src.llm.action_planner import ActionPlanner, Action

planner = ActionPlanner()

# Test scenarios
scenarios = [
    (IntentResult(primary_intent="generate", task_type="mcq"), None, "Tao 3 cau MCQ"),
    (IntentResult(primary_intent="generate", task_type="slide"), None, "Tao slide"),
    (IntentResult(primary_intent="explain"), None, "Giai thich mang LAN"),
    (IntentResult(primary_intent="chat"), None, "Xin chao"),
    (IntentResult(primary_intent="analyze"), None, "Xem diem"),
]

print("=" * 60)
print("[NODE 4] ActionPlanner")
print("=" * 60)
for intent, session, msg in scenarios:
    plan = planner.plan(intent, session, msg)
    print(f"  Message: '{msg}'")
    print(f"  Intent: {intent.primary_intent} | Task: {intent.task_type}")
    print(f"  -> Action: {plan.action.value} | Reason: {plan.reason}")
    print()

### 3.5 — Node: RAG Search (BM25 + Semantic + RRF -> Reranker)

In [ ]:
query = "Mang may tinh la gi va co nhung loai nao"

print("=" * 60)
print("[NODE 5] RAG Search")
print("=" * 60)

# 5a. CustomSearch (BM25 + Semantic + RRF)
t0 = time.time()
search_results = searcher.search(query, top_k=settings.RETRIEVER_TOP_K)
search_time = time.time() - t0
print(f"  CustomSearch: {len(search_results)} results ({search_time:.2f}s)")

# 5b. Reranker
t1 = time.time()
reranked = reranker.rerank(query, search_results, top_n=settings.RERANKER_TOP_N)
rerank_time = time.time() - t1
print(f"  Reranker: {len(reranked)} results ({rerank_time:.2f}s)")
print(f"  Total RAG: {search_time + rerank_time:.2f}s")
print()

# Show top chunks
print("  TOP CHUNKS:")
for i, r in enumerate(reranked):
    score = r.get('rerank_score', r.get('score', 0))
    preview = r['content'][:150].replace('\n', ' ')
    print(f"  [{i+1}] score={score:.4f}")
    print(f"      {preview}")
    print()

### 3.6 — Node: Handler (LLM Generation)

In [ ]:
from src.llm.handlers.chat_handler import ChatHandler
from src.llm.handlers.explain_handler import ExplainHandler
from src.llm.handlers.question.mcq_handler import MCQHandler
from src.llm.utils import format_contexts

context_text = format_contexts(reranked)  # From previous cell

print("=" * 60)
print("[NODE 6] Handler Execution")
print("=" * 60)

# --- 6a. ChatHandler ---
print("\n--- ChatHandler ---")
chat_h = ChatHandler()
t0 = time.time()
chat_resp = chat_h.handle(query="Mang may tinh la gi?", context=context_text)
print(f"  Time: {time.time()-t0:.2f}s | Length: {len(chat_resp)}")
print(f"  Preview: {chat_resp[:300]}")
print()

In [ ]:
# --- 6b. ExplainHandler ---
print("--- ExplainHandler ---")
explain_h = ExplainHandler()
t0 = time.time()
explain_resp = explain_h.handle(query="Giai thich mang LAN khac gi mang WAN", context=context_text)
print(f"  Time: {time.time()-t0:.2f}s | Length: {len(explain_resp)}")
print(f"  Preview: {explain_resp[:300]}")
print()

In [ ]:
# --- 6c. MCQHandler ---
print("--- MCQHandler ---")
mcq_h = MCQHandler()
t0 = time.time()
mcq_result = mcq_h.handle(query="Sinh 3 cau trac nghiem ve mang may tinh", context=context_text, num_questions=3)
gen_time = time.time() - t0
print(f"  Time: {gen_time:.2f}s")
if mcq_result:
    print(f"  Questions: {len(mcq_result.mcq)}")
    display = mcq_result.to_display_format()
    print(f"  Display:\n{display[:500]}")
else:
    print("  [!] Handler returned None")

### 3.7 — Node: Question Validator

In [ ]:
from src.llm.validators.question_validator import QuestionValidator

if mcq_result:
    print("=" * 60)
    print("[NODE 7] Question Validator")
    print("=" * 60)
    
    validator = QuestionValidator()
    t0 = time.time()
    val_result = validator.validate(
        question_type="mcq",
        context=context_text,
        questions_json=json.dumps(mcq_result.model_dump())
    )
    val_time = time.time() - t0
    
    print(f"  Time: {val_time:.2f}s")
    print(f"  All valid: {val_result.all_valid}")
    print(f"  Approved: {len(val_result.approved_questions)}")
    print(f"  Validations: {len(val_result.validations)}")
    for v in val_result.validations:
        print(f"    - {v}")
else:
    print("[!] Skip validator -- mcq_result is None")

---
## 4. Full Pipeline — Orchestrator.ask()

Chay toan bo pipeline end-to-end. Ket qua trace duoc ghi vao `pipeline_trace.log` dang **JSON**.

In [21]:
from src.llm.memory import MemoryManager

# Reset memory truoc moi test
orch.memory = MemoryManager()

QUERY = "tổng hợp kiến thức tin học của lơp 12"

print("=" * 70)
print(f"FULL PIPELINE -- Query: '{QUERY}'")
print("=" * 70)

t0 = time.time()
response_chunks = []
for chunk in orch.ask(QUERY):
    response_chunks.append(chunk)
total = time.time() - t0

full_response = "".join(response_chunks)
print(f"\n{'=' * 70}")
print(f"RESPONSE ({len(full_response)} chars, {total:.2f}s):")
print(f"{'=' * 70}")
print(full_response[:1000])
if len(full_response) > 1000:
    print(f"\n... [{len(full_response) - 1000} chars truncated]")

[22:36:28] INFO    | ============================================================
[22:36:28] INFO    | QUERY: 'tổng hợp kiến thức tin học của lơp 12'


FULL PIPELINE -- Query: 'tổng hợp kiến thức tin học của lơp 12'


[22:36:30] INFO    | IntentRouter: intent=explain, task_type=None, topic=Kiến thức Tin học lớp 12, is_new_topic=True
[22:36:30] INFO    | IntentRouter (2.21s): intent=explain, task_type=None, topic=Kiến thức Tin học lớp 12, is_new_topic=True
[22:36:30] INFO    | Topic changed: 'Hệ điều hành' -> 'Kiến thức Tin học lớp 12', creating new session
[22:36:30] INFO    | New session created: id=764117c5, topic='Kiến thức Tin học lớp 12', intent=explain
[22:36:30] INFO    | Session: id=764117c5, topic='Kiến thức Tin học lớp 12', msgs=0
[22:36:30] INFO    | ActionPlan: explain_concept (General concept explanation)
[22:37:24] INFO    | Total time: 56.06s
[22:37:24] INFO    | ============================================================



RESPONSE (5694 chars, 56.09s):
Dang tim tai lieu de giai thich...

Chào em, thầy là EduBot, trợ lý học tập Tin học THPT của em đây! Hôm nay, chúng ta sẽ cùng nhau tổng hợp lại những kiến thức Tin học quan trọng của lớp 12 nhé. Thầy sẽ giải thích thật chi tiết và dễ hiểu, bám sát theo sách giáo khoa để em nắm vững bài học.

---

### 1. Khái niệm cốt lõi

Lớp 12 là giai đoạn chúng ta đi sâu vào những ứng dụng thực tế của Tin học, đặc biệt là trong việc **xây dựng và quản lý thông tin trên mạng**, cũng như **lập trình để giải quyết các bài toán thực tế**.

---

### 2. Giải thích chi tiết

Trong chương trình Tin học lớp 12, chúng ta sẽ tập trung vào các mảng kiến thức chính sau:

*   **Xây dựng trang web với Google Sites:**
    *   **Mục đích:** Giúp em có thể tự tạo ra những trang web đơn giản, chia sẻ thông tin một cách trực quan và sinh động mà không cần biết quá nhiều về lập trình.
    *   **Cách thực hiện:** Em sẽ học cách chọn chủ đề phù hợp (như kỷ niệm về trường lớp, về gia đình, 

### 4.1 — Xem Pipeline Trace JSON

In [ ]:
print("=" * 70)
print("PIPELINE TRACE (pipeline_trace.log — JSON)")
print("=" * 70)
show_trace_log()

### 4.2 — Xem Debug Info (orchestrator.last_debug_info)

In [ ]:
print("=" * 70)
print("DEBUG INFO (orchestrator.last_debug_info)")
print("=" * 70)
show_debug_info(orch.last_debug_info)

### 4.3 — Xem App Log (n dong cuoi)

In [ ]:
print("=" * 70)
print("APP LOG (last 20 lines)")
print("=" * 70)
show_app_log(20)

---
## 5. Multi-Query Test — Chuoi hoi thoai

Test pipeline lien tiep nhieu query de kiem tra session management.

In [ ]:
# Reset memory
orch.memory = MemoryManager()

queries = [
    "Xin chao ban la ai",
    "Giai thich mang may tinh cho toi",
    "Tao 3 cau trac nghiem ve chu de nay",
]

print("=" * 70)
print("MULTI-QUERY TEST")
print("=" * 70)

for i, q in enumerate(queries, 1):
    print(f"\n{'=' * 70}")
    print(f"[Query {i}/{len(queries)}]: {q}")
    print(f"{'=' * 70}")
    
    t0 = time.time()
    chunks = list(orch.ask(q))
    total = time.time() - t0
    response = "".join(chunks)
    
    # Load JSON trace for this query
    trace_data = load_trace_json()
    steps = trace_data.get('steps', [])
    
    # Print summary per node
    for step in steps:
        node = step.get('node', '?')
        if node == 'ContextAnalyzer':
            print(f"  [1] ContextAnalyzer: enriched={step.get('enriched')}")
        elif node == 'IntentRouter':
            print(f"  [2] IntentRouter: intent={step.get('primary_intent')} | topic={step.get('topic')} | time={step.get('time_s')}s")
        elif node == 'SessionManager':
            print(f"  [3] SessionManager: id={step.get('session_id')} | topic={step.get('topic')} | msgs={step.get('total_messages')}")
        elif node == 'ActionPlanner':
            print(f"  [4] ActionPlanner: action={step.get('action')} | reason={step.get('reason')}")
        elif node == 'RAG':
            print(f"  [5] RAG: search={step.get('search_results')} | reranked={step.get('reranked')} | time={step.get('time_s')}s")
        elif node == 'Handler':
            print(f"  [6] Handler: action={step.get('action')} | status={step.get('status')} | {json.dumps({k:v for k,v in step.items() if k not in ('node','action','status')}, ensure_ascii=False)}")
    
    print(f"  Total: {total:.2f}s | Response: {len(response)} chars")
    print(f"  Preview: {response[:200]}")

### 5.1 — Xem Trace JSON cua query cuoi

In [ ]:
show_trace_log()

---
## 6. Custom Query — Tu nhap de test

In [ ]:
# === THAY DOI QUERY TAI DAY ===
CUSTOM_QUERY = "Tao 3 cau dung sai ve an toan thong tin"
# ===============================

print(f"Query: '{CUSTOM_QUERY}'")
print("=" * 70)

t0 = time.time()
chunks = list(orch.ask(CUSTOM_QUERY))
total = time.time() - t0
response = "".join(chunks)

print(f"\nResponse ({len(response)} chars, {total:.2f}s):")
print("=" * 70)
print(response[:1500])

print("\n" + "=" * 70)
print("PIPELINE TRACE JSON:")
print("=" * 70)
show_trace_log()

---
## 7. Timing Benchmark — Do thoi gian tung node

In [ ]:
# Reset memory
orch.memory = MemoryManager()

benchmark_query = "Tao 3 cau trac nghiem ve he dieu hanh"

print(f"Benchmark: '{benchmark_query}'")
print("=" * 70)

chunks = list(orch.ask(benchmark_query))

# Load JSON trace
trace_data = load_trace_json()
steps = trace_data.get('steps', [])

print(f"\n{'Node':<25} {'Time (s)':>10}")
print('=' * 37)

for step in steps:
    node = step.get('node', '?')
    # Find timing field
    t = step.get('time_s') or step.get('generation_time_s') or step.get('chat_time_s') or step.get('explain_time_s') or step.get('slide_time_s') or step.get('scorer_time_s')
    label = node
    if node == 'Handler':
        label = f"Handler:{step.get('action','?')}"
    if t is not None:
        bar = '#' * int(t * 2)
        print(f"  {label:<23} {t:>8.2f}s  {bar}")
    else:
        print(f"  {label:<23}      -")

total = trace_data.get('total_time_s', 0)
print('=' * 37)
print(f"  {'TOTAL':<23} {total:>8.2f}s")

# Response info
resp = trace_data.get('response', {})
print(f"\nResponse: {resp.get('length', 0)} chars")